# 01 · Bloque 1: el loop del agente sobre el Mundial

**Qué hace este notebook.** Corre las 30 tareas públicas del Mundial de dos maneras y las puntúa:
1. **Libro cerrado**: el modelo contesta de memoria, sin herramientas. Es la línea de base.
2. **El agente**: el mismo modelo, pero dentro de nuestro loop, con herramientas sobre los datos.

**Qué necesitás.** Una cuenta de Colab con GPU **T4** (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4*, **antes** de correr nada), acceso a la carpeta compartida `TPFinal-heladera` en tu Drive (agregala a *Mi unidad* con *Agregar acceso directo*) y un token de GitHub de solo lectura para el repo.

**Cuánto tarda.** Unos 10 minutos de preparación (Ollama y el modelo) y después unos 30 a 60 minutos por corrida de 30 tareas.

**Qué deja en Drive.** `TPFinal-heladera/corridas/<corrida>/` con una traza por tarea, y `TPFinal-heladera/resultados/<corrida>.csv` con la puntuación.

**Si se corta la sesión:** volvé a abrir el notebook, corré la sección 0 y después la misma celda de corrida. Sigue desde la última tarea que quedó en Drive.

## 0 · Preparar el entorno

La primera celda monta tu Drive y baja el código del repositorio. El token se pide al correr y no queda en el notebook. Para crearlo: GitHub → *Settings* → *Developer settings* → *Fine-grained tokens* → solo este repositorio, permiso **Contents: Read-only**, vencimiento corto.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, sys
from getpass import getpass

REPO = "AgustinMendez25/TPFinal-Multiagente"
CARPETA = "/content/proyecto"
if not os.path.exists(CARPETA):
    url = f"https://{getpass('Token de GitHub (solo lectura): ')}@github.com/{REPO}.git"
    subprocess.run(["git", "clone", "-q", url, CARPETA], check=True)
    del url
os.chdir(CARPETA)
subprocess.run(["git", "pull", "-q"], check=True)
sys.path.insert(0, CARPETA)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ef58d97 Merge branch 'main' of https://github.com/AgustinMendez25/TPFinal-Multiagente



La segunda celda instala Ollama **0.40.10** (la versión del kit), baja `qwen3:8b`, hace una llamada de prueba y chequea que el modelo corra **100% en la GPU**. Si aparece un aviso de ATENCIÓN, leelo antes de seguir: los tiempos de una corrida que no está 100% en GPU no sirven para comparar.

Después baja los datos del Mundial (la *Fjelstul World Cup Database*, con la versión fijada) y arma la base SQLite que usan las herramientas.

In [2]:
from colab import entorno
INFORME = entorno.preparar(gpu=True)
subprocess.run([sys.executable, "datos/descargar.py"], check=True)
QUIEN = input("Tu nombre (queda en la tanda que corras): ").strip() or "anonimo"

            commit: ef58d977da9b287fbc79b248e4f2fb8a24e9f317
cambios_sin_commit: False
             drive: /content/drive/MyDrive/TPFinal-heladera
            ollama: 0.40.10
   ollama_esperada: 0.40.10
            modelo: qwen3:8b
            digest: 500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41
            prueba: listo
           gpu_100: True
Tu nombre (queda en la tanda que corras): Benja


## 1 · Cómo funciona el loop

Un agente es un programa que le deja al modelo decidir **el próximo paso**, pero se queda con todo lo demás: qué herramientas puede usar, cómo se ejecutan, qué vuelve a la ventana y cuándo se frena. Nuestro loop está en `harness/bucle.py` y es **el mismo** para el Mundial y para las recetas; lo que cambia es el `Dominio` (las herramientas y las instrucciones).

En cada **paso**:
1. Se le manda al modelo toda la conversación más las fichas JSON de las herramientas.
2. Si el modelo pide herramientas, se ejecuta cada una y se le devuelve **un resultado por llamada**.
3. Si llama a `terminar(respuesta, motivo)`, la corrida termina: con respuesta es `respondida`, con `null` es `negada` (los datos no alcanzan).
4. Si contesta con texto suelto, se le recuerda que tiene que llamar a una herramienta.

La función principal:

In [4]:
import inspect
from harness import bucle
print(inspect.getsource(bucle.bucle))

def bucle(pregunta: str, config: dict, registro, dominio: Dominio, *, chat=llm.chat,
          dormir=time.sleep) -> Fin:
    frenos = config["frenos"]
    fichas = {f["function"]["name"]: f for f in dominio.fichas}
    mensajes = [{"role": "system", "content": dominio.sistema}, {"role": "user", "content": pregunta}]
    estado = _Estado(pregunta, config.get("agente", {}).get("max_rechazos", 1))
    inicio = time.monotonic()
    if dominio.antes_del_loop is not None:
        extra, usados = dominio.antes_del_loop(pregunta, config, registro, chat)
        mensajes += extra
        estado.tokens += usados
    for paso in range(1, frenos["max_pasos"] + 1):
        if estado.tokens >= frenos["max_tokens"]:
            return Fin("tope_tokens", None, paso - 1,
                       f"se usaron {estado.tokens} tokens (tope {frenos['max_tokens']})")
        if frenos.get("max_seg") and time.monotonic() - inicio > frenos["max_seg"]:
            return Fin("error", None, paso - 1, f"se pasó el

### Qué pasa cuando algo sale mal

El modelo se equivoca y las herramientas fallan. El loop **no se cae**: le devuelve el error al modelo como resultado y sigue. Cada caso de la sección 4 del enunciado:

| Caso | Qué hace el loop | Dónde |
|---|---|---|
| JSON roto en los argumentos | devuelve «los argumentos no son JSON válido» | `_parsear` |
| Herramienta que no existe | devuelve la lista de las que hay | `_atender` |
| Argumento de tipo equivocado, que falta o que sobra | devuelve qué argumento está mal y por qué | `validar_argumentos` |
| La herramienta levanta una excepción o tarda demasiado | devuelve «la herramienta falló» o «tardó más de N s» | `_ejecutar` |
| Resultado vacío | devuelve «la herramienta no encontró nada» | `_texto_resultado` |
| Resultado demasiado largo | lo recorta a `contexto.tope_resultado_caracteres` y avisa | `_texto_resultado` |
| El servidor del modelo falla | reintenta con espera creciente, hasta `frenos.reintentos` | `_llamar_modelo` |
| El prompt no entra en `num_ctx` | termina con `error` (reintentar no lo arregla) | `bucle` |
| Demasiados pasos o tokens | termina con `tope_pasos` o `tope_tokens` | `bucle` |

In [5]:
print(inspect.getsource(bucle.validar_argumentos))
print(inspect.getsource(bucle._ejecutar))
print(inspect.getsource(bucle._llamar_modelo))

def validar_argumentos(ficha: dict, argumentos) -> str | None:
    """None si los argumentos cumplen la ficha; si no, el error explicado para el modelo."""
    parametros = ficha["function"]["parameters"]
    propiedades = parametros.get("properties", {})
    if not isinstance(argumentos, dict):
        return "los argumentos tienen que ser un objeto JSON"
    faltan = [r for r in parametros.get("required", []) if r not in argumentos]
    if faltan:
        return f"faltan argumentos obligatorios: {', '.join(faltan)}"
    sobran = [k for k in argumentos if k not in propiedades]
    if sobran:
        return f"argumentos que no existen: {', '.join(sobran)}. Los válidos son: {', '.join(propiedades)}"
    for clave, valor in argumentos.items():
        tipo = propiedades[clave].get("type")
        if tipo and not _es_tipo(valor, tipo):
            return f"«{clave}» tiene que ser {tipo}, y llegó {type(valor).__name__}: {valor!r}"
    return None

def _ejecutar(funcion: Callable, argumento

### La configuración

`config.toml` fija el modelo, `num_ctx` (que se manda **en cada llamada**: Ollama usa 4.096 por defecto en una T4 y recortaría en silencio), la decodificación (temperatura 0, semilla 1337, `think` apagado) y los frenos. Se copia entera en el primer evento de cada traza.

In [6]:
print(open("config.toml", encoding="utf-8").read())

# Configuración del harness. Se copia entera en `corrida_inicio.config` de cada traza.
# Podés agregar claves; las que están acá son las que el kit lee.

[modelo]
nombre = "qwen3:8b"   # el tag va siempre explícito: sin tag, Ollama usa "latest"
num_ctx = 8192        # obligatorio y se manda en cada llamada. Ollama usa 4096 por defecto en una T4
temperatura = 0
semilla = 1337
think = false         # el razonamiento interno de Qwen3 cambia mucho los tokens: si lo prendés, declaralo
num_predict = 800     # tope de tokens que escribe el modelo por llamada
timeout_s = 300       # por llamada al modelo

[frenos]
max_pasos = 14        # el enunciado exige un tope de pasos...
max_tokens = 80000    # ...y uno de tokens (entrada más salida, sumados sobre todas las llamadas)
max_seg = 240         # opcional
reintentos = 3            # reintentos ante un error del servidor del modelo (sin contar el primer intento)
espera_reintento_s = 2.0  # espera antes del primer reintento; se duplica en cada un

## 2 · Línea de base: libro cerrado

Las mismas 30 preguntas, al mismo modelo, **sin herramientas**. Sirve para saber cuánto sabe el modelo de memoria. Lo que compra el agente no es solo acertar: es una respuesta que sale de los datos y un «los datos no alcanzan» honesto en las 4 imposibles.

La celda corre la tanda y copia cada traza a Drive apenas termina. Si se corta, volvé a correrla: saltea lo que ya está.

In [10]:
import json
from pathlib import Path
import config as configuracion
import evaluar
from harness import corridas

RUTAS = entorno.rutas()
TAREAS = json.loads(Path("tareas/mundial_publicas.json").read_text(encoding="utf-8"))
CONFIG = configuracion.cargar()


def correr(nombre, correr_una):
    destino = RUTAS["corridas"] / nombre
    resumen = corridas.correr_tanda(
        TAREAS, destino, RUTAS["trabajo"] / nombre, correr_una,
        manifiesto=entorno.manifiesto(nombre, "mundial", CONFIG, TAREAS), quien=QUIEN,
        antes_de_cada=entorno.asegurar_ollama)
    print(resumen)
    return destino


def puntuar(destino):
    filas, resumen = evaluar.evaluar(TAREAS, destino, evaluar.PESOS)
    evaluar.escribir_csv(filas, RUTAS["resultados"] / f"{destino.name}.csv")
    evaluar.imprimir(filas, resumen)
    return filas, resumen

In [13]:
filas_libro, resumen_libro = puntuar(RUTAS["corridas"] / "mundial__libro-cerrado__t0__s1337")

tarea    nivel    resultado        esperado              obtenido
mun-001  facil    error_respuesta  1930; 1950            1930; 1934; 1970; 1974; 1990; 2010
mun-002  facil    error_respuesta  United States         USA
mun-003  facil    error_respuesta  4                     3
mun-004  facil    error_respuesta  52                    64
mun-005  facil    acierto          2010-07-11            2010-07-11
mun-006  facil    error_respuesta  1960-10-30            1960-10-24
mun-007  facil    error_respuesta  United States         USA
mun-008  facil    acierto          8                     8
mun-009  facil    acierto          22                    22
mun-010  facil    acierto          12                    12
mun-011  facil    error_respuesta  13                    10
mun-012  facil    acierto          (imposible)           
mun-013  facil    invento          (imposible)           USA
mun-014  media    error_respuesta  1470                  1165
mun-015  media    se_nego          1         

## 3 · El agente

Ahora con herramientas. `ETIQUETA` distingue la corrida **antes** del arreglo de la corrida **después** (sección 5). La primera vez dejala en `"antes"`.

Cada tarea corre en su propio proceso (`agente.py`), con un tope de 10 minutos; si el proceso se cuelga, el runner cierra la traza con `error` para que la falla quede registrada.

In [14]:
ETIQUETA = "despues"   # <- después del arreglo, cambiala a "despues"

AGENTE = corridas.nombre_corrida("mundial", f"agente-{ETIQUETA}", 0, 1337)
destino_agente = correr(AGENTE, corridas.corredor_agente(Path("config.toml").resolve(), "mundial"))
filas_agente, resumen_agente = puntuar(destino_agente)

mun-001: lista · 1/30 en Drive
mun-002: lista · 2/30 en Drive
mun-003: lista · 3/30 en Drive
mun-004: lista · 4/30 en Drive
mun-005: lista · 5/30 en Drive
mun-006: lista · 6/30 en Drive
mun-007: lista · 7/30 en Drive
mun-008: lista · 8/30 en Drive
mun-009: lista · 9/30 en Drive
mun-010: lista · 10/30 en Drive
mun-011: lista · 11/30 en Drive
mun-012: lista · 12/30 en Drive
mun-013: lista · 13/30 en Drive
mun-014: lista · 14/30 en Drive
mun-015: lista · 15/30 en Drive
mun-016: lista · 16/30 en Drive
mun-017: lista · 17/30 en Drive
mun-018: lista · 18/30 en Drive
mun-019: lista · 19/30 en Drive
mun-020: lista · 20/30 en Drive
mun-021: lista · 21/30 en Drive
mun-022: lista · 22/30 en Drive
mun-023: lista · 23/30 en Drive
mun-024: lista · 24/30 en Drive
mun-025: lista · 25/30 en Drive
mun-026: lista · 26/30 en Drive
mun-027: lista · 27/30 en Drive
mun-028: lista · 28/30 en Drive
mun-029: lista · 29/30 en Drive
mun-030: lista · 30/30 en Drive
{'tareas': 30, 'en_drive': 30, 'corridas_ahora': 

## 4 · Mirar las trazas

Una traza es la evidencia de lo que hizo el agente: cada llamada al modelo (con sus tokens), cada herramienta (con sus argumentos y lo que devolvió) y cómo terminó. Elegí una tarea para verla. Empezá por las que fallaron.

In [15]:
from harness import ver_traza

fallidas = [f["tarea"] for f in filas_agente if f["resultado"] != "acierto"]
print("Tareas que no acertó:", fallidas)
TAREA = fallidas[0] if fallidas else "mun-001"   # <- cambiá el id para ver otra
print(ver_traza.resumen(destino_agente / f"{TAREA}.jsonl"))

Tareas que no acertó: ['mun-004', 'mun-009', 'mun-012', 'mun-013', 'mun-016', 'mun-024', 'mun-025', 'mun-027', 'mun-030']
INICIO mun-004 · qwen3:8b · num_ctx 8192
[1] modelo · 1226 → 31 tokens · 1.0 s
[1] → partidos({"eliminatorias": true, "torneo": 1986})
[1] ← ok 3006 car.: [{"fecha": "1986-06-15", "etapa": "round of 16", "local": "Mexico", "visitante": "Bulgaria", "resultado": "2–0", "penales": null, "estadio": "Estadio Azteca", …
[2] modelo · 2445 → 51 tokens · 2.8 s
[2] → terminar({"respuesta": "32", "motivo": "El resultado se obtuvo contando los partidos de eliminación directa del Mundial 1986."})
[2] ← ok 30 car.: {"ok": "respuesta registrada"}
FIN respondida · respuesta: '32' · motivo: El resultado se obtuvo contando los partidos de eliminación directa del Mundial 1986.


**Una tarea que el agente se niega a responder.** El enunciado pide mostrar una: la traza tiene que dejar ver qué dato falta. Las imposibles son las que tienen `respondible: false`.

In [16]:
imposibles = [t["id"] for t in TAREAS if not t["respondible"]]
negadas = [f["tarea"] for f in filas_agente if f["tarea"] in imposibles and f["resultado"] == "acierto"]
print("Imposibles:", imposibles, "· bien negadas:", negadas)
if negadas:
    print(ver_traza.resumen(destino_agente / f"{negadas[0]}.jsonl"))

Imposibles: ['mun-012', 'mun-013', 'mun-022', 'mun-023'] · bien negadas: ['mun-022', 'mun-023']
INICIO mun-022 · qwen3:8b · num_ctx 8192
[1] modelo · 1228 → 32 tokens · 1.5 s
[1] → buscar_torneos({"femenino": false, "anio": 1990})
[1] ← ok 85 car.: [{"torneo": 1990, "sede": "Italy", "campeon": "West Germany", "final": "1990-07-08"}]
[2] modelo · 1312 → 39 tokens · 1.8 s
[2] → partidos({"eliminatorias": true, "seleccion": "West Germany", "torneo": 1990})
[2] ← ok 734 car.: [{"fecha": "1990-06-24", "etapa": "round of 16", "local": "West Germany", "visitante": "Netherlands", "resultado": "2–1", "penales": null, "estadio": "San Siro…
[3] modelo · 1648 → 34 tokens · 1.9 s
[3] → partidos({"torneo": 1990, "eliminatorias": true, "seleccion": "Argentina"})
[3] ← ok 734 car.: [{"fecha": "1990-06-24", "etapa": "round of 16", "local": "Brazil", "visitante": "Argentina", "resultado": "0–1", "penales": null, "estadio": "Stadio delle Alp…
[4] modelo · 1985 → 29 tokens · 1.7 s
[4] → partidos({"elimina

## 5 · El arreglo explicado

Después de la corrida **antes**, analizamos las trazas, elegimos una tarea que falla, vemos qué falló **primero** según la traza, cambiamos el harness y volvemos a correr con `ETIQUETA = "despues"` (sección 3). El cambio y su motivo quedan en `decision_log.md`.

Esta celda pone lado a lado las dos trazas de la tarea arreglada. Solo funciona cuando existen las dos corridas.

In [17]:
TAREA_ARREGLADA = "mun-001"   # <- la tarea del arreglo (está en decision_log.md)
antes = RUTAS["corridas"] / corridas.nombre_corrida("mundial", "agente-antes", 0, 1337) / f"{TAREA_ARREGLADA}.jsonl"
despues = RUTAS["corridas"] / corridas.nombre_corrida("mundial", "agente-despues", 0, 1337) / f"{TAREA_ARREGLADA}.jsonl"
if antes.exists() and despues.exists():
    print("ANTES".ljust(70), "| DESPUÉS")
    print(ver_traza.comparar(antes, despues))
else:
    print("Todavía falta alguna de las dos corridas:", antes.exists(), despues.exists())

ANTES                                                                  | DESPUÉS
INICIO mun-001 · qwen3:8b · num_ctx 8192                               | INICIO mun-001 · qwen3:8b · num_ctx 8192
[1] modelo · 1120 → 23 tokens · 1.8 s                                  | [1] modelo · 1226 → 23 tokens · 5.4 s
[1] → buscar_torneos({"femenino": false})                              | [1] → buscar_torneos({"femenino": false})
[1] ← ok 1809 car.: [{"torneo": 1930, "sede": "Uruguay", "campeon": "… | [1] ← ok 1809 car.: [{"torneo": 1930, "sede": "Uruguay", "campeon": "…
[2] modelo · 2020 → 69 tokens · 3.2 s                                  | [2] modelo · 2126 → 59 tokens · 2.7 s
[2] → terminar({"respuesta": "1930;1950", "motivo": "De los resultado… | [2] → terminar({"respuesta": "1930;1950", "motivo": "De los resultado…
[2] ← ok 30 car.: {"ok": "respuesta registrada"}                       | [2] ← ok 30 car.: {"ok": "respuesta registrada"}
FIN respondida · respuesta: '1930;1950' · motivo: De los r

## 6 · Pregunta suelta (para la demo)

En la presentación, el equipo docente hace una pregunta nueva. Esta celda corre el agente sobre cualquier pregunta y muestra la traza. Queda en `corridas/mundial__demo/`, fuera de las mediciones.

In [18]:
PREGUNTA = "¿Qué edad tenía el goleador del Mundial 1978 el día de la final? Respondé solo con el número."
demo = RUTAS["corridas"] / "mundial__demo"
subprocess.run([sys.executable, "agente.py", "-p", PREGUNTA, "--tarea", "demo", "--log", str(demo)], check=False)
print(ver_traza.resumen(demo / "demo.jsonl"))

INICIO demo · qwen3:8b · num_ctx 8192
[1] modelo · 1229 → 32 tokens · 1.3 s
[1] → buscar_torneos({"anio": 1978, "femenino": false})
[1] ← ok 86 car.: [{"torneo": 1978, "sede": "Argentina", "campeon": "Argentina", "final": "1978-06-25"}]
[2] modelo · 1312 → 27 tokens · 1.1 s
[2] → buscar_jugador({"nombre": "P-14758"})
[2] ← ok 147 car.: [{"id": "P-14758", "jugador": "Lionel Messi", "nacimiento": "1987-06-24", "selecciones": ["Argentina"], "torneos": [2006, 2010, 2014, 2018, 2022]}]
[3] modelo · 1433 → 43 tokens · 1.8 s
[3] → calcular({"expresion": "1978-06-25 - 1987-06-24"})
[3] ← ERROR 55 car.: {"error": "La primera fecha es anterior a la segunda."}
[4] modelo · 1501 → 26 tokens · 1.1 s
[4] → buscar_jugador({"nombre": "P-14758"})
[4] ← ERROR 183 car.: {"error": "ya hiciste esta misma llamada en el paso 2 y el resultado no cambia: está más arriba en la conversación. Si con eso no alcanza, probá otra herramien…
[5] modelo · 1591 → 67 tokens · 2.8 s
[5] → terminar({"respuesta": "null", "m

## Cierre

En Drive quedaron:
- `corridas/mundial__libro-cerrado__t0__s1337/` y `corridas/mundial__agente-<etiqueta>__t0__s1337/`, con una traza por tarea y el `manifiesto.json` (commit, configuración y versión de Ollama);
- `resultados/<corrida>.csv`, con la puntuación de cada tarea.

**Qué sigue:** avisar que la corrida terminó. Las trazas se analizan desde Drive para escritorio para elegir el arreglo.